In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig, ResidueGroup, ScoringWeights

cfg = AnalysisConfig(
    ligand_resname  = "UNK",
    topology_glob   = "equilibrating_topology.pdb",
    trajectory_glob = "trajectory.xtc",
    dt_ns           = 2.0,
    output_dir      = Path("./hbond_results"),
    figures_dir     = Path("./figures"),
    residue_groups  = {
        # Example — replace with your pocket residues
        # "ECD": ResidueGroup(
        #     name="ECD",
        #     resids=[100, 150, 200],
        #     resnames=[],
        #     bonus=20.0,
        # ),
    },
    scoring = ScoringWeights(
        occ_weight=50.0,
        dist_weight=30.0,
        angle_weight=20.0,
        stability_weight=15.0,
    ),
)

REPLICA_ROOTS = [
    # Path("../run01"),
]

# Contact fingerprint target residues (e.g. binding pocket)
ECD_RESIDS: list[int] = []

SNAPSHOT_DIR = Path("./snapshots")
# ============================================================

In [ ]:
import pickle
import matplotlib.pyplot as plt
from mdatools.analysis.hbonds import run_hbond_batch
from mdatools.scoring.template_scorer import TemplateScorer
from mdatools.scoring.snapshot import SnapshotSelector
from mdatools.analysis.contacts import ContactFingerprint
from mdatools.universe import load_and_align
from mdatools.io.loaders import discover_replicas

cfg.make_dirs()
SNAPSHOT_DIR.mkdir(exist_ok=True)

# Load or recompute H-bond results
pkl_path = cfg.output_dir / "result_dfs_hbonds.pkl"
if pkl_path.exists():
    with open(pkl_path, "rb") as f:
        hbond_results = pickle.load(f)
    print(f"Loaded H-bond results from {pkl_path}")
else:
    hbond_results = run_hbond_batch(REPLICA_ROOTS, cfg)
    with open(pkl_path, "wb") as f:
        pickle.dump(hbond_results, f)

In [ ]:
# Score templates
scorer = TemplateScorer(cfg)
scored = scorer.score_all(hbond_results)
score_df = scorer.to_dataframe(scored)
score_df.to_csv(cfg.output_dir / "template_scores.csv", index=False)

print(score_df.to_string(index=False))

In [ ]:
from mdatools.plotting.scoring_plots import plot_score_breakdown, plot_hbond_quality_map

plot_score_breakdown(score_df, save_path=cfg.figures_dir / "template_score_breakdown.png")
plt.show()

plot_hbond_quality_map(score_df, save_path=cfg.figures_dir / "hbond_quality_map.png")
plt.show()

In [ ]:
# Extract best snapshots
replicas = discover_replicas(REPLICA_ROOTS, cfg)
selector = SnapshotSelector(cfg)

for rep in replicas:
    name = rep["name"]
    if name not in hbond_results or hbond_results[name].summary.empty:
        continue
    snap = selector.select(hbond_results[name])
    print(f"{name}: best frame={snap.frame}, dist={snap.distance}, angle={snap.angle}")
    u = load_and_align(rep["topology"], rep["trajectory"], cfg)
    pdb_path = SNAPSHOT_DIR / f"snapshot_{name}_frame{snap.frame:04d}.pdb"
    selector.extract_pdb(u, snap, pdb_path)
    print(f"  -> Saved: {pdb_path}")

In [ ]:
# Contact fingerprint for top replica
from mdatools.plotting.validation_plots import plot_contact_fingerprint

if scored:
    top_name = scored[0].sample_name
    top_rep = next(r for r in replicas if r["name"] == top_name)
    u = load_and_align(top_rep["topology"], top_rep["trajectory"], cfg)
    fp = ContactFingerprint(cfg)
    fp_df = fp.run(u, ecd_resids=ECD_RESIDS)
    fp_df.to_csv(cfg.output_dir / f"contact_fp_{top_name}.csv", index=False)

    plot_contact_fingerprint(
        fp_df, top_name,
        save_path=cfg.figures_dir / f"contact_fp_{top_name}.png"
    )
    plt.show()